In [47]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)
from utils.user_utils import get_clf_eval

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")

In [49]:
train['var3'].replace(-999999, 2, inplace=True)
test['var3'].replace(-999999, 2, inplace=True)

y = train['TARGET']
X = train.drop(['ID', 'TARGET'], axis=1)

X_test_only = test.drop(['ID'], axis=1)

In [50]:
# df.info()

# print("\n 결측값의 수:", df.isna().sum().sum())

# <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 76020 entries, 0 to 76019
# Columns: 369 entries, var3 to var38
# dtypes: float64(111), int64(258)
# memory usage: 214.0 MB

#  결측값의 수: 0

In [51]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Train shape: (60816, 369)
Validation shape: (15204, 369)


In [52]:
# -----------------------------
# 4. StandardScaler
# -----------------------------
scaler = StandardScaler()
scaler.fit(X_train)          # ✔ train만 fit

X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test_only)   # 여기서 test도 transform만!

In [53]:
# 레이블의 분포 확인
cust_cnt = y.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [83]:
# -----------------------------
# 5. 모델 학습: XGBoost
# -----------------------------

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=5, # Positive class(1)의 gradient를 키워서 1을 더 잘 학습하도록 돕는다
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train_scaled, y_train)
pred_val = xgb.predict(X_val_scaled)
proba_val = xgb.predict_proba(X_val_scaled)[:,1]

get_clf_eval(y_val, pred_val, proba_val)

best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (proba_val >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"\nBest Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# 최적 threshold로 성능 출력
pred_best = (proba_val >= best_threshold).astype(int)
get_clf_eval(y_val, pred_best, proba_val)


folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8497, 정확도: 0.9252, 정밀도: 0.2220, 재현율: 0.3555, F1: 0.2733
오차행렬:
[[13852   750]
 [  388   214]]

Best Threshold: 0.44, Best F1: 0.2999
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8497, 정확도: 0.9094, 정밀도: 0.2161, 재현율: 0.4900, F1: 0.2999
오차행렬:
[[13532  1070]
 [  307   295]]


In [76]:
# -----------------------------
# 6. RandomForest
# -----------------------------

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight={0:1, 1:2}, # positive class의 중요도를 올림
    random_state=0,
    n_jobs=-1
)

rf.fit(X_train, y_train)    # 랜포는 스케일링 필요 없음
rf_pred = rf.predict(X_val)
rf_proba = rf.predict_proba(X_val)[:, 1]

get_clf_eval(y_val, rf_pred, rf_proba)

for thr in thresholds:
    pred_thr = (rf_proba >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"\nBest Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# 최적 threshold로 성능 출력
pred_best = (rf_proba >= best_threshold).astype(int)
get_clf_eval(y_val, pred_best, rf_proba)


folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8264, 정확도: 0.9604, 정밀도: 0.0000, 재현율: 0.0000, F1: 0.0000
오차행렬:
[[14602     0]
 [  602     0]]

Best Threshold: 0.13, Best F1: 0.2974
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8264, 정확도: 0.8630, 정밀도: 0.1675, 재현율: 0.6196, F1: 0.2637
오차행렬:
[[12748  1854]
 [  229   373]]


In [57]:
# ============================================
# 6. TEST.CSV에 대한 최종 예측
#
# TARGET=1(불만족)을 얼마나 잘 잡아내는지
# 즉, “잠재적으로 문제가 생길 고객”을 잘 찾아내는지?
#
# 불만족(=1) 고객 비율이 너무 낮기 때문에
# F1 score과 recall이 낮게 나오는 것이 정상이고,
# AUC를 중심으로 보는 게 맞아.
# ============================================

# XGBoost test 예측값
xgb_test_pred = xgb.predict_proba(X_test_scaled)[:, 1]

# RandomForest test 예측값
rf_test_pred = rf.predict_proba(X_test_only)[:, 1]

print("\n===== FINAL TEST PREDICTIONS =====")
print("\nXGBoost Test Predictions (probability of TARGET=1(불만족)):")
print(xgb_test_pred[:20])   # 상위 20개만 미리보기

print("\nRandomForest Test Predictions (probability of TARGET=1(불만족)):")
print(rf_test_pred[:20])    # 상위 20개만 미리보기


===== FINAL TEST PREDICTIONS =====

XGBoost Test Predictions (probability of TARGET=1(불만족)):
[0.04930619 0.05511775 0.00103434 0.00724789 0.00128104 0.25067496
 0.01753684 0.18530677 0.02739658 0.01895448 0.02584288 0.00335761
 0.01247218 0.01089097 0.00569734 0.02837694 0.1344037  0.00233657
 0.01139107 0.01790423]

RandomForest Test Predictions (probability of TARGET=1(불만족)):
[0.0272395  0.02927176 0.01248364 0.05284314 0.01261565 0.12722314
 0.05366328 0.11728255 0.02126337 0.02232213 0.01857102 0.01593014
 0.02060549 0.01330972 0.01860971 0.02761147 0.07766145 0.01330972
 0.01330496 0.02024374]


In [ ]:
best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (proba_val >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"Best Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# 최적 threshold로 성능 출력
pred_best = (proba_val >= best_threshold).astype(int)
get_clf_eval(y_val, pred_best, proba_val)

# -----------------------------
# RandomForest Threshold Optimization for F1
# -----------------------------



print(f"[RF] Best Threshold: {best_threshold:.2f}, Best F1 Score: {best_f1:.4f}")

# 최적 threshold로 평가 지표 출력
rf_pred_best = (rf_proba >= best_threshold).astype(int)
get_clf_eval(y_val, rf_pred_best, rf_proba, model_name='RF_best')
